# 07 — On-device chatbot: LoRA fine-tune (master plan D2/D3)

Fine-tunes a small instruct model on the **Verifier-filtered grounded-QA dataset** built by
`scripts/generate_chat_dataset.py` (D1), targeting in-browser serving via WebLLM/MLC (D4).

**Status: ready-to-run scaffold — has NOT been executed yet.** Runs on Kaggle (T4/P100, free).

Inputs (attach as a Kaggle dataset):
- `chat_dataset.jsonl` — every row is `{messages:[system,user,assistant], meta:{...}}`,
  where the user turn contains the evidence block and the assistant answer cites `[ids]`.
  100% of rows passed the deterministic Verifier at generation time.

Open decision (master plan §13): **base model size** — `Qwen2.5-0.5B-Instruct` (fast in-browser,
~350 MB q4) vs `Qwen2.5-1.5B-Instruct` (stronger, ~950 MB q4). Default below is 0.5B; flip one line.

Acceptance gates (D3): held-out grounding ≥ 0.98 · refusal accuracy ≥ 0.9 · beats the
deterministic template on an LLM-judge win-rate. **Do not ship a model that misses a gate.**

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # §13 decision: or "Qwen/Qwen2.5-1.5B-Instruct"
DATA_PATH  = "/kaggle/input/autovaluate-chat-dataset/chat_dataset.jsonl"
OUT_DIR    = "/kaggle/working/autovaluate-assistant-lora"
SEED       = 42
MAX_LEN    = 2048          # evidence blocks are long; do not truncate the answer
EPOCHS     = 2
LR         = 1e-4

import os
USE_WANDB = bool(os.environ.get("WANDB_API_KEY"))  # WS 0.1 — optional, works without

In [ ]:
%pip install -q -U transformers peft trl datasets accelerate
if USE_WANDB:
    %pip install -q wandb

In [ ]:
# ── Data: JSONL -> chat-templated splits ────────────────────────────────────
import json, random
from datasets import Dataset

rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8")]
random.Random(SEED).shuffle(rows)
n_eval = max(64, int(0.1 * len(rows)))
eval_rows, train_rows = rows[:n_eval], rows[n_eval:]
print(f"train {len(train_rows)} · eval {len(eval_rows)} · "
      f"refusal share {sum(r['meta']['intent']=='refusal' for r in rows)/len(rows):.0%}")

train_ds = Dataset.from_list([{"messages": r["messages"]} for r in train_rows])
eval_ds  = Dataset.from_list([{"messages": r["messages"]} for r in eval_rows])

In [ ]:
# ── LoRA SFT ────────────────────────────────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

config = SFTConfig(
    output_dir=OUT_DIR, seed=SEED, max_length=MAX_LEN,
    num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine",
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    logging_steps=10, eval_strategy="epoch", save_strategy="epoch",
    report_to="wandb" if USE_WANDB else "none",
)

trainer = SFTTrainer(model=model, args=config, peft_config=peft_config,
                     train_dataset=train_ds, eval_dataset=eval_ds,
                     processing_class=tokenizer)
trainer.train()

In [ ]:
# ── Held-out grounding + refusal eval (D3 acceptance gates) ─────────────────
# Light twin of the deterministic Verifier: every AED figure the model writes must
# already appear in the evidence block of its own prompt. The authoritative gate at
# runtime remains the deterministic verifier (server) / its client twin (lib/assistant).
import re

NUM = re.compile(r"(\d[\d,]{2,})")

def figures(text):
    return {m.replace(",", "") for m in NUM.findall(text)}

def generate(messages):
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=200, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

grounded = refused = n_refusal = 0
for r in eval_rows:
    gen = generate(r["messages"])
    evidence = r["messages"][1]["content"]
    ungrounded = figures(gen) - figures(evidence)
    if not ungrounded:
        grounded += 1
    if r["meta"]["intent"] == "refusal":
        n_refusal += 1
        if not figures(gen):   # a correct refusal quotes no figure at all
            refused += 1

report = {
    "grounding_rate": grounded / len(eval_rows),
    "refusal_accuracy": (refused / n_refusal) if n_refusal else None,
    "gate_grounding": grounded / len(eval_rows) >= 0.98,
    "gate_refusal": (refused / n_refusal >= 0.9) if n_refusal else None,
}
print(json.dumps(report, indent=2))
assert report["gate_grounding"], "FAILED the 0.98 grounding gate — do not ship this adapter"

In [ ]:
# ── Save adapter (small) + merged model (for MLC/WebLLM conversion, D4) ─────
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
merged = trainer.model.merge_and_unload()
merged.save_pretrained(f"{OUT_DIR}-merged", safe_serialization=True)
tokenizer.save_pretrained(f"{OUT_DIR}-merged")
print("Adapter ->", OUT_DIR, "· merged ->", f"{OUT_DIR}-merged")

## Next steps (D4/D5 — after this notebook has run and passed its gates)

1. **Model card + upload** (WS 0.2): push adapter + merged model to Hugging Face Hub with the
   eval JSON above in the card. *Needs the HF account decision.*
2. **Quantize + convert for the browser** (D4): `mlc_llm convert_weight <merged> --quantization q4f16_1`
   → serve via WebLLM alongside the existing ONNX CV path; keep the client-side Verifier twin as
   the hard gate; fall back Groq/Gemini → template exactly as today.
3. **Tool-use** (D5): extend D1 generation with `/estimate` tool-call transcripts so what-if
   questions re-price instead of hedge.
4. **Honest comparison**: LLM-judge win-rate vs the deterministic template on held-out questions —
   ship only if the fine-tune actually wins.